In [4]:
import pandas as pd 
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

In [5]:
predict_conversion_df = pd.read_csv('digital_marketing_campaign_dataset.csv')
google_ads_df = pd.read_csv('GoogleAds_DataAnalytics_Sales_Uncleaned.csv')
marketing_and_product_df = pd.read_csv('marketing_and_product_performance.csv')

I decided to use the ColumnTransformer to be able to compare the four distance metrics against one another across all three datasets. 

In [8]:
from sklearn.model_selection import cross_val_score, train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.metrics import f1_score, make_scorer

# Define features and target
X = predict_conversion_df.drop(['CustomerID', 'Conversion'], axis=1)
y = predict_conversion_df['Conversion']

# Identify numeric and categorical columns
numeric_features = X.select_dtypes(include=['int64', 'float64']).columns
categorical_features = X.select_dtypes(include=['object']).columns

# Create preprocessing pipeline
preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_features),
        ('cat', OneHotEncoder(drop='first'), categorical_features)
    ])

metrics = ['euclidean', 'manhattan', 'minkowski', 'cosine']
results_acc = {}
results_f1 = {}

for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor),
                                 ('classifier', knn)])
    
    cv_acc = cross_val_score(pipeline, X, y, cv=5, scoring='accuracy')
    cv_f1 = cross_val_score(pipeline, X, y, cv=5, scoring='f1')
    results_acc[metric] = cv_acc.mean()
    results_f1[metric] = cv_f1.mean()

print(f"Majority class baseline: {y.value_counts(normalize=True).max():.4f}")
for metric in metrics:
    print(f"Metric: {metric:10} | Acc: {results_acc[metric]:.4f} | F1: {results_f1[metric]:.4f}")

Majority class baseline: 0.8765
Metric: euclidean  | Acc: 0.8778 | F1: 0.9340
Metric: manhattan  | Acc: 0.8773 | F1: 0.9336
Metric: minkowski  | Acc: 0.8778 | F1: 0.9340
Metric: cosine     | Acc: 0.8691 | F1: 0.9283


For predict_conversion_df, the Euclidean and Minkowski distance metrics brought the highest accuracy (roughly 0.8778) and F1 score (0.9340). Since the dataset is primarily continuous numerical features, it would make the most sense to use the Euclidean distance since it measures a straight distance in feature space. Manhattan distance came very close but slightly underperformed when compared to the previous two. The cosine was the worst performing metric, which makes sense as it is better suited for text data or sparse datasets since the magnitude of the vectors is less important than their orientation. 

In [27]:
# Further clean Location
location_map = {
    'hyderbad': 'hyderabad',
    'hydrebad': 'hyderabad',
    'hyderabad': 'hyderabad'
}
google_ads_df['Location'] = google_ads_df['Location'].replace(location_map)
X_ga.loc[:, 'Location'] = X_ga['Location'].replace(location_map)

print(X_ga['Location'].unique())

results_ga_final = {}
for metric in metrics:
    knn = KNeighborsClassifier(n_neighbors=5, metric=metric)
    pipeline = Pipeline(steps=[('preprocessor', preprocessor_ga),
                                 ('classifier', knn)])
    
    valid_idx = y_ga.notnull()
    X_clean = X_ga[valid_idx]
    y_clean = y_ga[valid_idx]
    
    cv_acc = cross_val_score(pipeline, X_clean, y_clean, cv=5, scoring='accuracy')
    results_ga_final[metric] = cv_acc.mean()

for metric, score in results_ga_final.items():
    print(f"Metric: {metric:10} | Acc: {score:.4f}")

C:\Users\santo\AppData\Local\Temp\ipykernel_11876\236719839.py:7: FutureWarning: ChainedAssignmentError: behaviour will change in pandas 3.0!
You are setting values through chained assignment. Currently this works in certain cases, but when using Copy-on-Write (which will become the default behaviour in pandas 3.0) this will never work to update the original DataFrame or Series, because the intermediate object on which we are setting values will behave as a copy.
A typical example is when you are setting values in a column of a DataFrame, like:

df["col"][row_indexer] = value

Use `df.loc[row_indexer, "col"] = values` instead, to perform the assignment in a single step and ensure this keeps updating the original `df`.

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy

  google_ads_df['Location'] = google_ads_df['Location'].replace(location_map)


['hyderabad']
Metric: euclidean  | Acc: 0.5586
Metric: manhattan  | Acc: 0.5542
Metric: minkowski  | Acc: 0.5586
Metric: cosine     | Acc: 0.5538


After cleaning categorical noise from the Location feature of google_ads_df, I was able to see that Euclidean and Minkowski distances had the best mean CV accuracy of 0.5586. This dataset is mainly concerned with continuous numerical values, which Euclidean is best suited for as it preserves the geomatric relationships between points in a dense feature space. Although the Euclidean and Minkowski distances are the best performing, they hold relatively low predictive power

In [37]:
from sklearn.neighbors import KNeighborsClassifier, KNeighborsRegressor

# 1. Classification: Predict "High Conversions" (Above Median)
y_mp_clf = (marketing_and_product_df['Conversions'] > marketing_and_product_df['Conversions'].median()).astype(int)
X_mp = marketing_and_product_df.drop([
    'Campaign_ID', 'Product_ID', 'Customer_ID', 'Flash_Sale_ID', 'Bundle_ID',
    'Conversions', 'Revenue_Generated', 'ROI', 'Units_Sold'
], axis=1)

# Categorical/Numerical column identification
numeric_mp = X_mp.select_dtypes(include=['int64', 'float64']).columns
categorical_mp = X_mp.select_dtypes(include=['object']).columns

# Standard Pipeline Setup
preprocessor_mp = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_mp),
        ('cat', OneHotEncoder(drop='first', handle_unknown='ignore'), categorical_mp)
    ])

print("--- KNN Classification Results (Target: High Conversions) ---")
results_mp_clf = {}
for metric in ['euclidean', 'manhattan', 'minkowski', 'cosine']:
    pipe_clf = Pipeline(steps=[('preprocessor', preprocessor_mp),
                                ('classifier', KNeighborsClassifier(n_neighbors=5, metric=metric))])
    acc = cross_val_score(pipe_clf, X_mp, y_mp_clf, cv=5, scoring='accuracy').mean()
    results_mp_clf[metric] = acc
    print(f"Metric: {metric:10} | Accuracy: {acc:.4f}")

# 2. Regression: Predict "Conversions" count
y_mp_reg = marketing_and_product_df['Conversions']

print("\n--- KNN Regression Results (Target: Conversions) ---")
results_mp_reg = {}
for metric in ['euclidean', 'manhattan', 'minkowski', 'cosine']:
    pipe_reg = Pipeline(steps=[('preprocessor', preprocessor_mp),
                                ('regressor', KNeighborsRegressor(n_neighbors=5, metric=metric))])
    r2 = cross_val_score(pipe_reg, X_mp, y_mp_reg, cv=5, scoring='r2').mean()
    results_mp_reg[metric] = r2
    print(f"Metric: {metric:10} | R2 Score: {r2:.4f}")


--- KNN Classification Results (Target: High Conversions) ---
Metric: euclidean  | Accuracy: 0.5056
Metric: manhattan  | Accuracy: 0.5074
Metric: minkowski  | Accuracy: 0.5056
Metric: cosine     | Accuracy: 0.5044

--- KNN Regression Results (Target: Conversions) ---
Metric: euclidean  | R2 Score: -0.1872
Metric: manhattan  | R2 Score: -0.1939
Metric: minkowski  | R2 Score: -0.1872
Metric: cosine     | R2 Score: -0.1947


For marketing_and_product_df the results showed that the Manhattan distance is best for classification with an accuracy of 0.5074, while Euclidean and Minkowski distances were best for regression with an R^2 score of -0.1872. Manhattan's robustness to outliers likely gives it an edge in classification, whereas Euclidean's straight-line calculation captures the variance better in a regression context.